# Laboratorio 06.1 — Chunking y metadatos

Fragmentaremos documentos largos en unidades recuperables y mediremos cómo el tamaño y el solapamiento modifican cantidad, longitud y preservación de contexto.

## 1. Objetivos de aprendizaje

Al finalizar podrás:
- Fragmentar documentos con tamaño y solapamiento configurables.
- Conservar identificadores y metadatos de origen en cada fragmento.
- Comparar tamaños de fragmento manteniendo otros parámetros constantes.
- Explicar el intercambio entre contexto, precisión de búsqueda e índice.

## 2. Conceptos

Un fragmento es una unidad que puede indexarse y recuperarse. Fragmentos muy grandes mezclan temas y pueden añadir contexto irrelevante; fragmentos pequeños pueden separar una afirmación de su contexto. Un solapamiento ayuda a conservar continuidad entre ventanas, pero aumenta el índice y puede duplicar evidencia. Este laboratorio cuenta palabras, no tokens reales del modelo.

## 3. Arquitectura

```mermaid
flowchart LR
    A[Documento fuente] --> B[Division en ventanas]
    B --> C[Fragmentos con metadatos]
    C --> D[Estadisticas de fragmentacion]
    C --> E[Vectorizador y prueba de recuperacion]
    E --> F[Contexto top-k]
```

## 4. Preparación del entorno

Usamos el corpus FAQ compartido, la función `chunk_text` reutilizable y pandas. La división se explica en el código para que el solapamiento sea visible.

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

from llm_engineering.environment import preparar_entorno
from llm_engineering.evaluation.metrics import hit_rate_at_k
from llm_engineering.retrieval.chunking import chunk_text
from llm_engineering.retrieval.vector_search import rank_by_cosine

raiz_curso = preparar_entorno()
documentos = pd.read_json(raiz_curso / 'data' / 'documents' / 'faq.jsonl', lines=True).to_dict(orient='records')

## 5. Definición del problema

El corpus contiene ocho respuestas FAQ con identificadores y temas. Buscaremos qué configuración produce fragmentos que retengan información útil para una consulta sobre el plazo de devolución, manteniendo trazabilidad al documento original.

## 6. Línea base

La línea base trata cada documento completo como una sola unidad. Esto preserva todo su contexto, pero puede devolver más texto del necesario y no permite aislar pasajes breves.

In [ ]:
consulta = 'devolucion durante 30 dias desde entrega'
print(f'Documentos completos: {len(documentos)}')
print(f'Palabras promedio por documento: {sum(len(d["texto"].split()) for d in documentos) / len(documentos):.1f}')

## 7. Implementación

El helper divide por palabras y valida que el solapamiento sea menor que el tamaño. Envolvemos cada texto resultante con `documento_id`, tema e índice para poder rastrear su fuente.

In [ ]:
def fragmentar_corpus(corpus: list[dict[str, str]], tamano: int, solapamiento: int) -> list[dict[str, object]]:
    """Fragmenta cada documento y conserva metadatos de origen."""
    fragmentos = []
    for documento in corpus:
        partes = chunk_text(documento['texto'], chunk_size=tamano, overlap=solapamiento)
        for indice, texto_fragmento in enumerate(partes):
            fragmentos.append({
                'id': f"{documento['id']}_chunk_{indice}",
                'documento_id': documento['id'],
                'tema': documento['tema'],
                'indice': indice,
                'texto': texto_fragmento,
            })
    return fragmentos

fragmentos_ejemplo = fragmentar_corpus(documentos, tamano=12, solapamiento=2)
display(pd.DataFrame(fragmentos_ejemplo).head(8))

## 8. Experimentos

Variamos tamaño de fragmento y dejamos fijo el solapamiento en dos palabras. Registramos cantidad, longitud media, duplicación aproximada y fuente preservada.

In [ ]:
filas_configuracion = []
for tamano in (8, 12, 20):
    fragmentos = fragmentar_corpus(documentos, tamano=tamano, solapamiento=2)
    filas_configuracion.append({
        'tamano_palabras': tamano,
        'solapamiento': 2,
        'numero_fragmentos': len(fragmentos),
        'palabras_promedio': sum(len(f['texto'].split()) for f in fragmentos) / len(fragmentos),
        'documentos_con_metadatos': len({f['documento_id'] for f in fragmentos}),
    })
tabla_chunking = pd.DataFrame(filas_configuracion)
display(tabla_chunking)
tabla_chunking.to_csv(raiz_curso / 'outputs' / '06_01_chunking_experiments.csv', index=False)

## 9. Evaluación

Además de tamaño, comparamos si al menos un fragmento relevante del documento aparece entre los tres primeros resultados. Esta prueba usa TF-IDF como baseline y mantiene igual vectorizador dentro de cada configuración; la calidad de generación no se evalúa aquí.

In [ ]:
filas_recuperacion = []
documento_relevante = 'devolucion_plazo'
for tamano in (8, 12, 20):
    fragmentos = fragmentar_corpus(documentos, tamano=tamano, solapamiento=2)
    vectorizador = TfidfVectorizer()
    matriz = vectorizador.fit_transform([f['texto'] for f in fragmentos]).toarray()
    vector_consulta = vectorizador.transform([consulta]).toarray()[0]
    ranking = rank_by_cosine(vector_consulta, matriz, top_k=3)
    ids_documentos_top = [fragmentos[indice]['documento_id'] for indice, _ in ranking]
    filas_recuperacion.append({
        'tamano_palabras': tamano,
        'hit_rate_at_3': hit_rate_at_k(ids_documentos_top, {documento_relevante}, 3),
        'primer_documento': ids_documentos_top[0],
    })
evaluacion_chunking = pd.DataFrame(filas_recuperacion)
display(evaluacion_chunking)
evaluacion_chunking.to_csv(raiz_curso / 'outputs' / '06_01_chunking_retrieval.csv', index=False)

## 10. Discusión

El tamaño óptimo depende de la longitud de las respuestas, límites del modelo y estrategia de recuperación. Más fragmentos pequeños pueden mejorar localización, pero generan más vectores y más duplicados. Conserva siempre el documento de origen para citar, depurar y aplicar controles sobre contenido no confiable.

## 11. Ejercicios

1. Cambia solo el solapamiento manteniendo fijo el tamaño.
2. Prueba un documento largo con una respuesta que cruza el límite entre fragmentos.
3. Añade número de sección y posición inicial a los metadatos.
4. Compara fragmentación por caracteres con fragmentación por palabras.

## 12. Desafío

Diseña una evaluación de chunking con consultas etiquetadas y varios tamaños. Optimiza Hit Rate y costo de contexto simultáneamente, y explica cómo resolverías resultados que requieren contexto de dos fragmentos vecinos.

## 13. Ideas clave

- Chunk size y overlap cambian el índice y el contexto recuperado.
- Los metadatos de origen permiten rastrear evidencia.
- Evalúa recuperación por separado de generación.
- Un helper por palabras no equivale a chunking por tokens del modelo.